In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import anndata as ad

# Single cell

In [ ]:
# Path to your .h5ad file
file_path = "/data/projects/manuela/aging/scRNA_aging-mouse/annotated-aging-soupxed.h5ad"

# Load the AnnData object
adata = sc.read_h5ad(file_path)
# Print basic information about the data
print(adata)
adata.obs["sample"]

In [ ]:
import scanpy as sc
import pandas as pd

genes_of_interest = ["Qpct", "Qpctl", "Ccl2"]

# 1. Check if genes exist in adata.var_names
present_genes = [gene for gene in genes_of_interest if gene in adata.var_names]
print("Genes present:", present_genes)

# 2. Extract expression for genes of interest
# This assumes adata.X is raw or normalized expression matrix
expr_data = adata[:, present_genes].X

# Convert to DataFrame if sparse matrix for easy processing
if hasattr(expr_data, "toarray"):
    expr_data = expr_data.toarray()

expr_df = pd.DataFrame(expr_data, columns=present_genes, index=adata.obs_names)

# 3. Add sample and cell type info to expression dataframe
expr_df["sample"] = adata.obs["sample"]
expr_df["celltype"] = adata.obs["celltype"]

# 4. Group by sample and cell type and calculate mean expression
mean_expr = expr_df.groupby(["sample", "celltype"])[present_genes].mean()

print(mean_expr)

In [ ]:
adata.obs

In [ ]:
import scanpy as sc

genes_of_interest = ["Qpct", "Qpctl", "Ccl2"]
present_genes = [gene for gene in genes_of_interest if gene in adata.var_names]

# Dotplot across sample groups
sc.pl.dotplot(adata, var_names=present_genes, groupby="sample", swap_axes=True, standard_scale="var", 
              title="Gene Expression in Samples")

# Dotplot across celltype groups
sc.pl.dotplot(adata, var_names=present_genes, groupby="celltype", swap_axes=True, standard_scale="var", 
              title="Gene Expression in Cell Types")

# Check raw singlecell files

In [ ]:
import scanpy as sc

file_paths = [
    "/data/projects/manuela/aging/scRNA_aging-mouse/sg18_soupx.h5ad",
    "/data/projects/manuela/aging/scRNA_aging-mouse/sg20_soupx.h5ad",
    "/data/projects/manuela/aging/scRNA_aging-mouse/sg24_soupx.h5ad",
    "/data/projects/manuela/aging/scRNA_aging-mouse/sg26_soupx.h5ad",
    "/data/projects/manuela/aging/scRNA_aging-mouse/sg28_soupx.h5ad",
]

genes_of_interest = ["Qpct", "Qpctl", "Ccl2"#, "Mcb1"
                    ]

In [ ]:
subset_adatas = []

for file_path in file_paths:
    adata = sc.read_h5ad(file_path, backed='r')
    present_genes = [gene for gene in genes_of_interest if gene in adata.var_names]
    
    if present_genes:
        print(f"File {file_path} contains genes: {present_genes}")
        
        gene_idx = [adata.var_names.get_loc(g) for g in present_genes]
        expr_subset = adata.X[:, gene_idx].toarray() if hasattr(adata.X, "toarray") else adata.X[:, gene_idx]
        
        obs_df = adata.obs[['orig.ident']].copy()
        var_df = adata.var.loc[present_genes].copy()
        
        adata_subset = ad.AnnData(X=expr_subset, obs=obs_df, var=var_df)
        
        # Add a column to identify the sample source for concatenation labeling
        adata_subset.obs['sample_batch'] = adata_subset.obs['orig.ident'].astype(str)
        subset_adatas.append(adata_subset)
    else:
        print(f"File {file_path} contains none of the genes {genes_of_interest}")

# Concatenate all in-memory subsets
adata_concat = ad.concat(subset_adatas, join='inner', label='batch', keys=[f'file{i}' for i in range(len(subset_adatas))])

# Normalize and log-transform
sc.pp.normalize_total(adata_concat, target_sum=1e4)
sc.pp.log1p(adata_concat)

# Plot gene expression comparison dotplot grouped by sample_batch or celltype
sc.pl.dotplot(adata_concat, var_names=genes_of_interest, groupby='sample_batch', standard_scale='var', title='Gene Expression by Sample Batch')

# Preprocessing Scanpy Pipeline

In [ ]:
import scanpy as sc
import anndata as ad

file_paths = [
    "/data/projects/manuela/aging/scRNA_aging-mouse/sg18_soupx.h5ad",
    "/data/projects/manuela/aging/scRNA_aging-mouse/sg20_soupx.h5ad",
    "/data/projects/manuela/aging/scRNA_aging-mouse/sg24_soupx.h5ad",
    "/data/projects/manuela/aging/scRNA_aging-mouse/sg26_soupx.h5ad",
    "/data/projects/manuela/aging/scRNA_aging-mouse/sg28_soupx.h5ad",
]

genes_of_interest = ["Qpct", "Qpctl"]
subset_adatas = []

for file_path in file_paths:
    adata = sc.read_h5ad(file_path, backed='r')
    present_genes = [gene for gene in genes_of_interest if gene in adata.var_names]
    
    if present_genes:
        print(f"File {file_path} contains genes: {present_genes}")
        
        gene_idx = [adata.var_names.get_loc(g) for g in present_genes]
        expr_subset = adata.X[:, gene_idx].toarray() if hasattr(adata.X, "toarray") else adata.X[:, gene_idx]
        
        obs_df = adata.obs[['orig.ident']].copy()
        var_df = adata.var.loc[present_genes].copy()
        
        adata_subset = ad.AnnData(X=expr_subset, obs=obs_df, var=var_df)
        adata_subset.obs['sample_batch'] = adata_subset.obs['orig.ident'].astype(str)
        subset_adatas.append(adata_subset)
    else:
        print(f"File {file_path} contains none of the genes {genes_of_interest}")

# Concatenate subsets
adata_concat = ad.concat(subset_adatas, join='inner', label='batch', keys=[f'file{i}' for i in range(len(subset_adatas))])

# Make an in-memory copy of the concatenated AnnData for full processing
adata_in_memory = adata_concat.copy()

# Full preprocessing pipeline
sc.pp.normalize_total(adata_in_memory, target_sum=1e4)
sc.pp.log1p(adata_in_memory)
sc.pp.highly_variable_genes(adata_in_memory, n_top_genes=2000, subset=True)
sc.pp.scale(adata_in_memory, max_value=10)
sc.tl.pca(adata_in_memory, svd_solver='arpack')
sc.pp.neighbors(adata_in_memory, n_neighbors=10, n_pcs=40)
sc.tl.leiden(adata_in_memory)
sc.tl.umap(adata_in_memory)

# Clustering visualization
sc.pl.umap(adata_in_memory, color=['leiden', 'sample_batch'])

# Gene expression dotplot on clustered data
sc.pl.dotplot(adata_in_memory, var_names=genes_of_interest, groupby='sample_batch', standard_scale='var',
              title='Gene Expression by Sample Batch')
